In [2]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [3]:
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [8]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

list_of_files = glob.glob('data/csv/*.csv')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = pd.read_csv(f, sep=',', header=0, index_col=False)
    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values()).reset_index()

# Drop unnecessary columns
final_df = final_df.drop(columns=['year_of_prediction', 'realization_year', 'index'])

# normalize precipitation values
final_df['precip'] = final_df['precip']/30

In [9]:
# Calculating all the statistics for each region and model
# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction', 'lead_time'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_5'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction', 'lead_time']).agg(['mean', 'std']).reset_index()
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'lead_time', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction', 'lead_time'], 
                        right_on=['region', 'model', 'season', 'month_of_prediction', 'lead_time'], how='left').dropna()

# Calculating metrics
stat_clean['potential_skill'] = np.square(stat_clean['corr'])
stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

# drop unnecessary columns
stat_clean = stat_clean.drop(columns=['corr', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std'])
 
# Save the final DataFrame to a CSV file
stat_clean.to_csv('data/csv/metrics/stat_clean.csv', index=False)

ValueError: Length mismatch: Expected axis has 11 elements, new values have 9 elements

In [44]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='potential_skill')
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    # Create the heatmap and force y ticklabels to remain visible
    ax = sns.heatmap(
        d,
        vmin=0, vmax=0.5,
        cmap=sns.color_palette('Reds', 10),
        fmt=".2f",
        linewidths=0.1,
        linecolor='black',
        square=True,
        yticklabels=True  # ensure ticklabels are drawn
    )
    
    # Set tick label properties explicitly
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

    ax.invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/potential_skill.png')
plt.close()

In [ ]:
'''# keep first 3 months of prediction of each model for each region and season
def keep_months_of_prediction(df, category):
    temp = df.copy()
    if category == 'short':
        for season in temp['season'].unique():
            if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
            & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
            # keep short lead (0-1) of prediction of each model for each region and season
            max = temp.loc[temp['season'] == season, 'month_of_prediction'].max()
            temp.loc[(temp['season'] == season)] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= (max - 1))]
    elif category == 'medium':
        for season in temp['season'].unique():
            if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
            & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
            # keep medium category months (2-3) of prediction of each model for each region and season
            max = temp.loc[temp['season'] == season, 'month_of_prediction'].max()
            temp.loc[(temp['season'] == season)] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] <= (max - 2)) & (temp['month_of_prediction'] >= (max - 3))]
    elif category == 'long':
        for season in temp['season'].unique():
            if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
            & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
            # keep last 3 months (4-6) of prediction of each model for each region and season
            max = temp.loc[temp['season'] == season, 'month_of_prediction'].max()
            temp.loc[(temp['season'] == season)] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] <= (max - 4))]
    else:
        print("Invalid Category")
        return temp.dropna()
    return temp.dropna()'''


In [ ]:
'''
# SMME with MME cutoff
merged_mod = potential_skill.merge(an_bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
merged_mod = merged_mod.dropna()

# keep models that are higher than MME cutoff
MME_metrics = merged_mod[merged_mod['model'] != 'SMME']
MME_metrics['region_season'] = MME_metrics['region'] + " | " + MME_metrics['season'] # compile region and season into one column, split by " | "
MME_metrics = MME_metrics.drop(columns = ['region', 'season'])
MME_metrics = MME_metrics.groupby(['region_season'])[['model', 'potential_skill', 'an_agreement', 'bn_agreement']]
'''

In [ ]:
'''
SMME_dict = {}
for region_season, df in MME_metrics:
    region = region_season[0].split(" | ")[0]
    season = region_season[0].split(" | ")[1]
    potential_skill = df.loc[df['model'] == 'MME', 'potential_skill'].values[0]
    an_agreement = df.loc[df['model'] == 'MME', 'an_agreement'].values[0]
    bn_agreement = df.loc[df['model'] == 'MME', 'bn_agreement'].values[0]
    print(df)
    print(potential_skill)
    df = df[(df['potential_skill'] > potential_skill)]
    df.loc[:,'region_season'] = region_season[0]
    SMME_dict[region_season] = df

SMME_mod = pd.concat(SMME_dict.values(), ignore_index=True)
'''

      model  potential_skill  an_agreement  bn_agreement
0     CCSM4         0.093756      0.480519      0.467532
2     CESM1         0.158559      0.544156      0.500000
4      CMCC         0.244237      0.583333      0.574074
6   CanESM5         0.235913      0.500000      0.568254
8       DWD         0.165803      0.492424      0.533333
10    ECMWF         0.136601      0.545455      0.454545
12     GEM5         0.217061      0.600000      0.533333
14     GFDL         0.235886      0.506494      0.584416
16      JMA         0.137327      0.496970      0.450000
18    METEO         0.115215      0.516667      0.600000
20      MME         0.248303      0.610390      0.610390
22     NASA         0.097957      0.519481      0.428571
24     NCEP         0.110227      0.467532      0.454545
0.2483027759417765


ValueError: cannot set a frame with no defined index and a scalar

In [38]:
grouped = smme_df.groupby('region')
for region, region_df in grouped:
    region_name = str(region)
    model = 'SMME'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

In [45]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='conditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Conditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/conditional_bias.png')
plt.close()

In [46]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='unconditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Unconditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/unconditional_bias.png')
plt.close()